In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"

import pypto
import torch
import torch_npu
import numpy as np

本节的解答思路参考了 [《动手学深度学习》习题解答](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/ch02/ch02?id=_21-%e6%95%b0%e6%8d%ae%e6%93%8d%e4%bd%9c) ，并在此基础上补充了 PyPTO 的实现。  

### 练习2.1.1 
运行本节中的代码。将本节中的条件语句 `X == Y` 更改为 `X < Y` 或 `X > Y`，然后看看你可以得到什么样的张量。

**解答：**
将 `X == Y` 语句改为 `X < Y` 或 `X > Y`，运行代码后，会返回和 X 大小相同的张量，对于每个位置，如果 X 的当前位置元素小于（大于）Y 的当前位置元素，则新张量中相应项的值为 `True`， 反之为 `False`。

In [6]:
X = torch.arange(12, dtype=torch.float32).reshape((3,4))
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
X < Y,X > Y

(tensor([[ True, False,  True, False],
         [False, False, False, False],
         [False, False, False, False]]),
 tensor([[False, False, False, False],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True]]))

PyPTO 不支持直接使用运算符对张量进行操作。用 `pypto.lt(X, Y)` 替代 `X < Y`，用 `pypto.gt(X, Y)` 替代 `X > Y`。

In [2]:
@pypto.frontend.jit
def lt_kernel(X: pypto.Tensor([3, 4], pypto.DT_FP32),
              Y: pypto.Tensor([3, 4], pypto.DT_FP32),
              out: pypto.Tensor([3, 4], pypto.DT_BOOL)): 
    pypto.set_vec_tile_shapes(3, 8)
    out[:] = pypto.lt(X, Y)

@pypto.frontend.jit
def gt_kernel(X: pypto.Tensor([3, 4], pypto.DT_FP32),
              Y: pypto.Tensor([3, 4], pypto.DT_FP32),
              out: pypto.Tensor([3, 4], pypto.DT_BOOL)): 
    pypto.set_vec_tile_shapes(3, 8)
    out[:] = pypto.gt(X, Y)

X_npu = torch.arange(12, dtype=torch.float32).reshape(3, 4).npu()
Y_npu = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]], dtype=torch.float32).npu()
out1_npu = torch.empty([3, 4], dtype=torch.bool).npu()
out2_npu = torch.empty([3, 4], dtype=torch.bool).npu()

lt_kernel(X_npu, Y_npu, out1_npu)
gt_kernel(X_npu, Y_npu, out2_npu)

print("X < Y:\n", out1_npu.cpu().numpy())
print("X > Y:\n", out2_npu.cpu().numpy())

X < Y:
 [[ True False  True False]
 [False False False False]
 [False False False False]]
X > Y:
 [[False False False False]
 [ True  True  True  True]
 [ True  True  True  True]]


### 练习2.1.2
用其他形状（例如三维张量）替换广播机制中按元素操作的两个张量。结果是否与预期相同？

**解答：**
* 能否得到预期结果要看两个张量的 shape 是否满足广播机制。
* 广播机制条件
  1. 每个张量至少有一个维度。
  2. 广播时, 从右往左对比两个张量的 shape, 需满足 shape 一样且大小匹配或 shape 不一样且大小不匹配的位置任意有一个张量为 1 或 shape 不一样且不匹配位置任意有一个张量不存在。

此处为 shape 一样且大小匹配。

In [4]:
a = torch.arange(6).reshape((3, 2, 1))
b = torch.arange(6).reshape((3, 2, 1))
a, b, a+b

(tensor([[[0],
          [1]],
 
         [[2],
          [3]],
 
         [[4],
          [5]]]),
 tensor([[[0],
          [1]],
 
         [[2],
          [3]],
 
         [[4],
          [5]]]),
 tensor([[[ 0],
          [ 2]],
 
         [[ 4],
          [ 6]],
 
         [[ 8],
          [10]]]))

此处为 shape 不一样且大小不匹配的位置任意有一个张量为 1。

In [5]:
a = torch.arange(3).reshape((1, 3, 1))
b = torch.arange(9).reshape((3, 1, 3))
a, b, a + b

(tensor([[[0],
          [1],
          [2]]]),
 tensor([[[0, 1, 2]],
 
         [[3, 4, 5]],
 
         [[6, 7, 8]]]),
 tensor([[[ 0,  1,  2],
          [ 1,  2,  3],
          [ 2,  3,  4]],
 
         [[ 3,  4,  5],
          [ 4,  5,  6],
          [ 5,  6,  7]],
 
         [[ 6,  7,  8],
          [ 7,  8,  9],
          [ 8,  9, 10]]]))

此处为 shape 不一样且不匹配位置任意有一个张量不存在

In [6]:
a = torch.arange(3).reshape((1, 3))
b = torch.arange(9).reshape((3, 1, 3))
a, b, a+b

(tensor([[0, 1, 2]]),
 tensor([[[0, 1, 2]],
 
         [[3, 4, 5]],
 
         [[6, 7, 8]]]),
 tensor([[[ 0,  2,  4]],
 
         [[ 3,  5,  7]],
 
         [[ 6,  8, 10]]]))

以下是使用 PyPTO 实现上述加法运算的参考代码，此处两个张量的 shape 相同且大小匹配：


In [3]:
@pypto.frontend.jit
def broad_add_sameshape_kernel(a: pypto.Tensor([3, 2, 1], pypto.DT_FP32),
          b: pypto.Tensor([3, 2, 1], pypto.DT_FP32),
          out: pypto.Tensor([3, 2, 1], pypto.DT_FP32)): 
        pypto.set_vec_tile_shapes(3, 2, 1)
        out[:] = pypto.add(a, b)
        
a_npu = torch.arange(6, dtype=torch.float32).reshape(3, 2, 1).npu()
b_npu = torch.arange(6, dtype=torch.float32).reshape(3, 2, 1).npu()
out_npu = torch.empty([3, 2, 1], dtype=torch.float32).npu()

broad_add_sameshape_kernel(a_npu, b_npu, out_npu)

print("tensor\n", a_npu.cpu().numpy())
print("tensor\n", b_npu.cpu().numpy())
print("tensor\n", out_npu.cpu().numpy())

tensor
 [[[0.]
  [1.]]

 [[2.]
  [3.]]

 [[4.]
  [5.]]]
tensor
 [[[0.]
  [1.]]

 [[2.]
  [3.]]

 [[4.]
  [5.]]]
tensor
 [[[ 0.]
  [ 2.]]

 [[ 4.]
  [ 6.]]

 [[ 8.]
  [10.]]]


此处为 shape 不一样且大小不匹配的位置任意有一个张量为 1。  

In [3]:
@pypto.frontend.jit
def broad_add__difshape_kernel(a: pypto.Tensor([3, 3, 1], pypto.DT_FP32),
          b: pypto.Tensor([3, 1, 3], pypto.DT_FP32),
          out: pypto.Tensor([3, 3, 3], pypto.DT_FP32)): 
        pypto.set_vec_tile_shapes(3, 3, 8)
        out[:] = pypto.add(a, b)
        
a_npu = torch.arange(9, dtype=torch.float32).reshape(3, 3, 1).npu()
b_npu = torch.arange(9, dtype=torch.float32).reshape(3, 1, 3).npu()
out_npu = torch.empty([3, 3, 3], dtype=torch.float32).npu()

broad_add__difshape_kernel(a_npu, b_npu, out_npu)

print("tensor\n", a_npu.cpu().numpy())
print("tensor\n", b_npu.cpu().numpy())
print("tensor\n", out_npu.cpu().numpy())

tensor
 [[[0.]
  [1.]
  [2.]]

 [[3.]
  [4.]
  [5.]]

 [[6.]
  [7.]
  [8.]]]
tensor
 [[[0. 1. 2.]]

 [[3. 4. 5.]]

 [[6. 7. 8.]]]
tensor
 [[[ 0.  1.  2.]
  [ 1.  2.  3.]
  [ 2.  3.  4.]]

 [[ 6.  7.  8.]
  [ 7.  8.  9.]
  [ 8.  9. 10.]]

 [[12. 13. 14.]
  [13. 14. 15.]
  [14. 15. 16.]]]


<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; line-height: 1.65; color: #4b5563; font-size: 14px; background-color: #ffffff;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;">本示例中，张量 a、b 的形状与上方 PyTorch 示例中的不完全一致，这是因为 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">pypto.add</code> 仅支持沿单一维度进行广播，而上方 PyTorch 示例中的张量则需要沿多个维度同时广播。关于 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">pypto.add</code> 的更多细节，可参阅 PyPTO 官方文档中<a href="https://gitcode.com/cann/pypto/blob/0.2.0/docs/api/operation/pypto-add.md" style="color: #2563eb; text-decoration: underline; font-weight: 500;">pypto.add</a>的内容。</li>
    </ul>
  </div>
</details>

此处为 shape 不一样且不匹配位置任意有一个张量不存在。

In [2]:
@pypto.frontend.jit
def broad_add_kernel(a: pypto.Tensor([1, 3], pypto.DT_FP32),
          b: pypto.Tensor([3, 1, 3], pypto.DT_FP32),
          out: pypto.Tensor([3, 1, 3], pypto.DT_FP32)): 
        pypto.set_vec_tile_shapes(3, 1, 8)
        out[:] = pypto.add(a, b)
        
a_npu = torch.arange(3, dtype=torch.float32).reshape(1, 3).npu()
b_npu = torch.arange(9, dtype=torch.float32).reshape(3, 1, 3).npu()
out_npu = torch.zeros([3, 1, 3], dtype=torch.float32).npu()

broad_add_kernel(a_npu, b_npu, out_npu)

print("tensor\n", a_npu.cpu().numpy())
print("tensor\n", b_npu.cpu().numpy())
print("tensor\n", out_npu.cpu().numpy())

tensor
 [[0. 1. 2.]]
tensor
 [[[0. 1. 2.]]

 [[3. 4. 5.]]

 [[6. 7. 8.]]]
tensor
 [[[ 0.  2.  4.]]

 [[ 3.  5.  7.]]

 [[ 6.  8. 10.]]]
